# Data extraction 

On this notebook I will extract the data from my zenodo database to only keep the exact documents ParlaMint uses.

In [1]:
pip install congreso

Note: you may need to restart the kernel to use updated packages.


In [2]:
from congreso import congreso as c 
import pandas as pd
from matplotlib import pyplot as plt

#terms = ["X", "XIV", "XIII", "XII", "XI"]
#t = c.load_jsons(terms)

In [3]:
import os
import gzip
import json

def load_local_jsons(roman_numerals: list[str], folder_path: str = "../json.gz"):
    dict_ = {}
    for numeral in roman_numerals:
        filename = os.path.join(folder_path, f'{numeral}.json.gz')
        if not os.path.exists(filename):
            print(f"File not found: {filename}")
            continue
        with gzip.open(filename, 'rt', encoding='UTF-8') as f:
            dict_[numeral] = json.load(f)
    return dict_


In [5]:
terms = ["VII", "X", "XIV","XV"]
t = load_local_jsons(terms)


Now that the data is extracted we just need to use some simple functions to be able to filter the documents on congreso.db. This filters will filter by year, diario de sesiones, section and finaly organ.

In [6]:
display(c.get_all_fields(t["X"]))

['ndia',
 'desu2',
 'id',
 'fecha_mensaje',
 'fecha',
 'secc',
 'desu1',
 'encabezado',
 'desu',
 'legislatura',
 'orga',
 'pdf_url',
 'texto',
 'mensaje',
 'subi',
 'seri',
 'cve',
 'numdoc']

In [7]:
def filter_DS_documents(term_dictionary):
  DS_term = []
  for doc in term_dictionary:
    if doc["encabezado"] == "DS":
      DS_term.append(doc)
  return DS_term



In [8]:
def filter_docs(docs, year):
    lowerbound = f"{year}0101"
    upperbound = f"{year}1231"
    # Filter documents based on the date range
    docs = filter_DS_documents(c.get_documents_interval_dates(docs, lowerbound, upperbound))
    
    # Filter by section
    filter_1 = []
    for doc in docs:
        if doc["secc"] in ["CONGRESO", "CONGRESO DE LOS DIPUTADOS"]:
            filter_1.append(doc)

    # Filter by organization
    filtered_docs = []
    for doc in filter_1:
        if doc["orga"] == "Pleno":
            filtered_docs.append(doc)

    # Sort by date
    filtered_docs = sorted(filtered_docs, key=lambda x: x['fecha'])
    
    # Return the filtered and sorted documents
    return filtered_docs


Finally, we now can get our filtered documents

In [9]:
# 2003: falls into term VII
d_2003 = filter_docs(t["VII"], "2003")

# 2013: falls into term X
d_2013 = filter_docs(t["X"], "2013")

# 2023: spans terms XIV and XV
d_2023_XIV = filter_docs(t["XIV"], "2023")
d_2023_XV = filter_docs(t["XV"], "2023")
d_2023 = d_2023_XIV + d_2023_XV


In [10]:
print("Docs 2003 filtered =",c.num_docs_term(d_2003))
print("Docs 2013 filtered =",c.num_docs_term(d_2013))
print("Docs 2014 filtered =",c.num_docs_term(d_2023))


Docs 2003 filtered = 84
Docs 2013 filtered = 79
Docs 2014 filtered = 46


Now I save the data in json format

In [11]:
import json
import os

# Create the folder if it doesn't exist
os.makedirs("clean-data", exist_ok=True)

dicts_by_year = {
    2003: d_2003,
    2013: d_2013,
    2023: d_2023,
}

for year, doc_dict in dicts_by_year.items():
    new_file = f"clean-data/d_{year}.json"
    with open(new_file, "w", encoding="utf-8") as f:
        json.dump(doc_dict, f, ensure_ascii=False, indent=4)
    print(f"Saved: {new_file}")



Saved: clean-data/d_2003.json
Saved: clean-data/d_2013.json
Saved: clean-data/d_2023.json


## Dump into MySQL

In [12]:
import pandas as pd
import json

def json_to_csv_for_mysql(json_path, csv_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        docs = json.load(f)

    df = pd.DataFrame(docs)

    # Ensure only columns present in the MySQL table
    expected_columns = [
        "pdf_url", "cve", "fecha", "fecha_mensaje", "mensaje", "ndia", "numdoc",
        "orga", "seri", "subi", "texto", "secc", "legislatura", "desu", "desu1", "desu2"
    ]
    for col in expected_columns:
        if col not in df.columns:
            df[col] = None  # fill missing columns

    df = df[expected_columns]
    df.to_csv(csv_path, index=False, quoting=1)  # quoting=1 means quote all strings
    print(f"Saved {csv_path}")

# Convert all three datasets
json_to_csv_for_mysql("clean-data/d_2003.json", "documents_2003.csv")
json_to_csv_for_mysql("clean-data/d_2013.json", "documents_2013.csv")
json_to_csv_for_mysql("clean-data/d_2023.json", "documents_2023.csv")


Saved documents_2003.csv
Saved documents_2013.csv
Saved documents_2023.csv
